# Analiza aktualnych trendów w PubMed dotyczących stwardnienia rozsianego

In [ ]:
!pip install biopython

## Pobieranie artykułów naukowych z PubMed
Definiujemy funkcję `fetch_pubmed_abstracts`, która za pomocą modułu `Entrez` wyszukuje i pobiera najnowsze publikacje dotyczące stwardnienia rozsianego. Dane są zapisywane bezpośrednio do pliku CSV.

In [ ]:
import time
import csv
from Bio import Entrez
Entrez.email = "example@com"
def fetch_pubmed_abstracts(query, max_results, batch_size, output_csv):
    search_handle = Entrez.esearch(db="pubmed", term=query, retmax=max_results, sort="pub date")
    search_results = Entrez.read(search_handle)
    search_handle.close()
    id_list = search_results["IdList"]
    articles_data = []

    for start in range(0, len(id_list), batch_size):
        batch_ids = id_list[start:start+batch_size]
        fetch_handle = Entrez.efetch(db="pubmed", id=batch_ids, retmode="xml")
        fetch_results = Entrez.read(fetch_handle)
        fetch_handle.close()

        for article in fetch_results.get('PubmedArticle', []):
            medline = article['MedlineCitation']
            pmid = medline['PMID']
            article_info = medline['Article']
            title = article_info.get('ArticleTitle', '')

            abstract_text = ""
            if 'Abstract' in article_info:
                abstract_parts = article_info['Abstract'].get('AbstractText', [])
                abstract_text = " ".join([str(part) for part in abstract_parts])

            articles_data.append({
                'PMID': str(pmid),
                'Title': str(title),
                'Abstract': abstract_text
            })

        time.sleep(1)

    with open(output_csv, 'w', newline='', encoding='utf-8') as csvfile:
        writer = csv.DictWriter(csvfile, fieldnames=['PMID', 'Title', 'Abstract'])
        writer.writeheader()
        writer.writerows(articles_data)

fetch_pubmed_abstracts("Multiple Sclerosis", 1000, 200, "ms_abstracts.csv")

## Czyszczenie i przygotowanie tekstu
Zaimplementowany poniżej proces oczyszcza teksty pobranych tytułów oraz abstraktów z nadmiarowych spacji przed analizą NLP.

In [ ]:
import pandas as pd
import nltk
from nltk.corpus import stopwords
import re

nltk.download('stopwords')
stop_words = set(stopwords.words('english'))

def clean_text(text):
    if not isinstance(text, str):
        return ""

input_file = "ms_abstracts.csv"
output_file = "ms_abstracts_cleaned.csv"
df = pd.read_csv(input_file)
df['Title'] = df['Title'].apply(clean_text)
df['Abstract'] = df['Abstract'].apply(clean_text)
df.to_csv(output_file, index=False, encoding='utf-8')

## Modelowanie tematyczne (Topic Modeling)
Używamy wyspecjalizowanego modelu osadzeń biomedycznych (`S-PubMedBert-MS-MARCO`) oraz algorytmów UMAP i HDBSCAN do automatycznego wykrycia i nazwania trendów (tematów) w publikacjach.

In [ ]:
!pip install bertopic

In [ ]:
import pandas as pd
from sklearn.feature_extraction.text import CountVectorizer, ENGLISH_STOP_WORDS
from sentence_transformers import SentenceTransformer
from umap import UMAP
from hdbscan import HDBSCAN
from bertopic import BERTopic
from bertopic.representation import KeyBERTInspired

df = pd.read_csv("/content/ms_abstracts.csv", dtype={"PMID": str})

df_valid = df.dropna(subset=['Abstract']).copy()
df_valid['Abstract'] = df_valid['Abstract'].astype(str)
df_valid = df_valid[df_valid['Abstract'].str.strip() != ""]

docs = (df_valid['Title'].fillna('') + ". " + df_valid['Abstract']).tolist()
pmids = df_valid['PMID'].tolist()
# print(len(docs))

embedding_model = SentenceTransformer("pritamdeka/S-PubMedBert-MS-MARCO")

MS_STOPWORDS = [
    "ms", "multiple", "sclerosis", "patient", "patients", "pwms", "disease", "diseases",
    "study", "studies", "result", "results", "conclusion", "conclusions", "method", "methods",
    "background", "objective", "objectives", "aim", "aims", "use", "used", "using",
    "clinical", "associated", "association", "compared", "analysis", "data",
    "significant", "significantly", "year", "years", "group", "groups", "included",
    "including", "showed", "found", "increased", "higher", "lower", "level", "levels",
]
vectorizer_model = CountVectorizer(
    stop_words=list(ENGLISH_STOP_WORDS | set(MS_STOPWORDS)),
    ngram_range=(1, 2),
    token_pattern=r"(?u)\b[a-zA-Z][a-zA-Z0-9]*(?:-[a-zA-Z0-9]+)*\b",
)

umap_model = UMAP(n_neighbors=15, n_components=5, min_dist=0.0, metric="cosine", random_state=42)
hdbscan_model = HDBSCAN(min_cluster_size=10, metric="euclidean",
                        cluster_selection_method="eom", prediction_data=True)

topic_model = BERTopic(
    embedding_model=embedding_model,
    umap_model=umap_model,
    hdbscan_model=hdbscan_model,
    vectorizer_model=vectorizer_model,
    representation_model=KeyBERTInspired(),
)
topics, probs = topic_model.fit_transform(docs)

df_valid['Topic'] = topics
df_valid[['PMID', 'Topic', 'Title', 'Abstract']].to_csv("/content/ms_topics_assigned.csv", index=False)
print(df_valid[['PMID', 'Topic']].head())
topic_info = topic_model.get_topic_info()
outliers = (df_valid['Topic'] == -1).mean()
print(topic_info[['Topic', 'Count', 'Representation']].to_string(max_colwidth=80))

In [ ]:
print(topic_info['Name'])

## Ekstrakcja leków z tematów i weryfikacja w RxNorm
Algorytm ekstrahuje substancje chemiczne z każdego klastra przy użyciu modelu NER, a następnie wysyła zapytania do bazy danych `NIH RxNorm (RxNav)` w celu standaryzacji i zachowania tylko autentycznych, zweryfikowanych nazw leków. Wykorzystujemy do tego biomedyczny model NER `en_ner_bc5cdr_md`, który specjalizuje się w automatycznym rozpoznawaniu nazw leków i substancji chemicznych w tekstach medycznych.

In [ ]:
!pip install spacy
!pip install https://s3-us-west-2.amazonaws.com/ai2-s2-scispacy/releases/v0.5.4/en_ner_bc5cdr_md-0.5.4.tar.gz --no-deps

In [ ]:
# import os
# import re
# import en_ner_bc5cdr_md

# model_dir = os.path.dirname(en_ner_bc5cdr_md.__file__)
# config_path = os.path.join(model_dir, "en_ner_bc5cdr_md-0.5.4", "config.cfg")

# print(f"Naprawiam plik: {config_path}")

# with open(config_path, "r", encoding="utf-8") as f:
#     config_data = f.read()

# config_data = re.sub(r"include_static_vectors\s*=\s*['\"]?True['\"]?", "include_static_vectors = true", config_data)

# with open(config_path, "w", encoding="utf-8") as f:
#     f.write(config_data)

# print("Gotowe! Plik konfiguracyjny został naprawiony.")

In [ ]:
!pip install spacy requests pandas
!pip install --no-deps https://s3-us-west-2.amazonaws.com/ai2-s2-scispacy/releases/v0.5.4/en_ner_bc5cdr_md-0.5.4.tar.gz

## Popularność leków
Ta sekcja sprawdza, jak często dany lek jest wymieniany w poszczególnych tematach badawczych. Lek jest zliczany na podstawie liczby abstraktów, w których pojawia się podczas omawiania danego tematu.

In [ ]:
import re
import time
from collections import Counter
from functools import lru_cache

import pandas as pd
import requests
import spacy


def is_valid_candidate(text: str) -> bool:
    if re.search(r"[±:<>]", text):
        return False
    if len(re.findall(r"[a-zA-Z]", text)) < 2:
        return False
    return True


@lru_cache(maxsize=None)
def verify_in_rxnorm(term: str) -> str | None:
    base_url = "https://rxnav.nlm.nih.gov/REST"
    time.sleep(0.05)

    try:
        resp = requests.get(
            f"{base_url}/rxcui.json",
            params={"name": term, "search": 1},
            timeout=5,
        )
        rxcuis = resp.json().get("idGroup", {}).get("rxnormId", [])
        if not rxcuis:
            return None

        rxcui = rxcuis[0]
        prop_resp = requests.get(f"{base_url}/rxcui/{rxcui}/properties.json", timeout=5)
        canonical_name = prop_resp.json().get("properties", {}).get("name")
        return canonical_name.lower() if canonical_name else term.lower()

    except requests.RequestException:
        return None


def extract_drugs_from_topics(df, text_column="Abstract_Original", topic_column="Topic"):
    nlp = spacy.load("en_ner_bc5cdr_md")

    topic_drugs_summary = []
    grouped = df.groupby(topic_column)

    for topic, group in grouped:
        if topic == -1:
            continue

        texts = [t for t in group[text_column] if isinstance(t, str) and t.strip()]
        all_drugs_in_topic = []

        for doc in nlp.pipe(texts, batch_size=50):
            drugs_in_this_publication = set()
            for ent in doc.ents:
                if ent.label_ != "CHEMICAL":
                    continue

                raw_text = ent.text.strip().lower()
                if not is_valid_candidate(raw_text):
                    continue

                verified_name = verify_in_rxnorm(raw_text)
                if verified_name:
                    drugs_in_this_publication.add(verified_name)

            all_drugs_in_topic.extend(drugs_in_this_publication)

        most_common_drugs = Counter(all_drugs_in_topic).most_common(10)

        topic_drugs_summary.append({
            "Topic": topic,
            "Top_Drugs": ", ".join([f"{drug} ({count})" for drug, count in most_common_drugs]),
            "Raw_Drug_List": [drug for drug, count in most_common_drugs],
        })

    return pd.DataFrame(topic_drugs_summary)

df = df_valid

results_df = extract_drugs_from_topics(df, text_column="Abstract", topic_column="Topic")
print(results_df[["Topic", "Top_Drugs"]])

## Wizualizacja danych
Wygenerowane wykresy przedstawiają strukturę tematów: rozkład wielkości poszczególnych klastrów oraz mapę ciepła (heatmap) pokazującą częstotliwość występowania kluczowych leków w określonych tematach badawczych.

In [ ]:
import re
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

BLUE = "#2a78d6"
TEXT, TEXT_2, GRID, EMPTY = "#0b0b0b", "#52514e", "#e5e4e0", "#f1f0ed"
SEQ_BLUES = ["#cde2fb", "#9ec5f4", "#6da7ec", "#3987e5", "#256abf", "#184f95", "#0d366b"]
plt.rcParams.update({
    "figure.dpi": 120, "font.size": 10, "axes.edgecolor": GRID, "axes.labelcolor": TEXT_2,
    "xtick.color": TEXT_2, "ytick.color": TEXT, "axes.titlesize": 12, "axes.titleweight": "bold",
    "axes.titlelocation": "left", "axes.spines.top": False, "axes.spines.right": False,
})


def topic_label(row, n_words=3):
    rep = row["Name"]
    words = rep if isinstance(rep, list) else str(rep).split(", ")
    return f"{row['Topic']}: " + ", ".join(w for w in words[:n_words] if w)


def parse_top_drugs(text):
    return {d.strip(): int(n) for d, n in re.findall(r"([^,]+?) \((\d+)\)", str(text))}


info = topic_info[topic_info["Topic"] != -1].copy()
info["label"] = info.apply(topic_label, axis=1)
n_outliers = int(topic_info.loc[topic_info["Topic"] == -1, "Count"].sum())
size = info.set_index("Topic")["Count"]

drugs = (results_df.assign(drugs=results_df["Top_Drugs"].map(parse_top_drugs))
         .set_index("Topic")["drugs"])
counts = pd.DataFrame(drugs.tolist(), index=drugs.index)
counts = counts.apply(lambda col: np.minimum(col, size.loc[counts.index]))
share = counts.div(size.loc[counts.index], axis=0) * 100


d = info.sort_values("Count")
fig, ax = plt.subplots(figsize=(9, 0.42 * len(d) + 1.4))
ax.barh(d["label"], d["Count"], color=BLUE, height=0.62)
for y, v in enumerate(d["Count"]):
    ax.text(v + d["Count"].max() * 0.01, y, str(v), va="center", fontsize=9, color=TEXT_2)
ax.set_title(f"Tematy w {int(topic_info['Count'].sum())} najnowszych publikacjach o SM", pad=22)
ax.text(0, 1.015, f"liczba publikacji w temacie; poza tematami (outliery): {n_outliers}",
        transform=ax.transAxes, color=TEXT_2, fontsize=9)
ax.spines["left"].set_visible(False)
ax.tick_params(axis="y", length=0)
ax.xaxis.grid(True, color=GRID, linewidth=0.8)
ax.set_axisbelow(True)
plt.tight_layout()
plt.show()


TOP_N_DRUGS = 12
top_drugs = counts.sum().sort_values(ascending=False).head(TOP_N_DRUGS).index
order = size.loc[counts.index].sort_values(ascending=False).index
m = share.loc[order, top_drugs]
c = counts.loc[order, top_drugs]

cmap = plt.matplotlib.colors.LinearSegmentedColormap.from_list("blues", SEQ_BLUES)
cmap.set_bad(EMPTY)
fig, ax = plt.subplots(figsize=(1.0 + 0.62 * len(top_drugs), 1.4 + 0.5 * len(order)))
im = ax.imshow(np.ma.masked_invalid(m.values.astype(float)), cmap=cmap, vmin=0,
               vmax=np.nanmax(m.values), aspect="auto")
for i in range(m.shape[0]):
    for j in range(m.shape[1]):
        if not np.isnan(m.iat[i, j]):
            ax.text(j, i, int(c.iat[i, j]), ha="center", va="center", fontsize=8,
                    color="white" if m.iat[i, j] > 0.55 * np.nanmax(m.values) else TEXT)
ax.set_xticks(range(len(top_drugs)), top_drugs, rotation=40, ha="right")
ax.set_yticks(range(len(order)), info.set_index("Topic").loc[order, "label"])
ax.set_xticks(np.arange(-0.5, len(top_drugs)), minor=True)
ax.set_yticks(np.arange(-0.5, len(order)), minor=True)
ax.grid(which="minor", color="white", linewidth=2)
ax.tick_params(which="both", length=0)
for s in ax.spines.values():
    s.set_visible(False)
cb = fig.colorbar(im, ax=ax, shrink=0.8, pad=0.02)
cb.set_label("% publikacji tematu", color=TEXT_2)
cb.outline.set_visible(False)
ax.set_title("Leki w tematach", pad=22)
ax.text(0, 1.015, "liczba w komórce = publikacje z lekiem; szare = lek spoza top 10 tematu",
        transform=ax.transAxes, color=TEXT_2, fontsize=9)
plt.tight_layout()
plt.show()

## Generowanie podsumowań tematów przez LLM (Groq)
Wykorzystujemy model LLM w chmurze Groq do wygenerowania podsumowań dla każdego tematu badawczego, uwzględniając najczęstsze leki oraz główny mianownik naukowy grupy badań.

In [ ]:
!pip install groq

In [ ]:
import pandas as pd
import time
from groq import Groq

API_KEY = "key"
client = Groq(api_key=API_KEY)
MODEL_NAME = "openai/gpt-oss-120b"

def generate_topic_summary(abstracts, top_drugs, topic_id, topic_name, keywords):

    combined_abstracts = "\n\n---\n\n".join(abstracts[:12])

    prompt = f"""Jesteś analitykiem literatury medycznej specjalizującym się w stwardnieniu rozsianym (Multiple Sclerosis).
Temat badawczy, z którego analizujesz abstrakty, to: {topic_name}.
Oto lista najczęstszych leków zidentyfikowanych w tym temacie badawczym: {top_drugs}.
Kieruj się słowami kluczowymi tematu(zawartymi od najważniejszego): {keywords}.

Poniżej znajdują się abstrakty naukowe przypisane do tego klastra tematycznego.
Na ich podstawie napisz zwięzłe i profesjonalne podsumowanie tego tematu po polsku (około 2-3 akapitów).

Twoje podsumowanie musi odpowiedzieć na te pytania:
1. Jaki jest główny, wspólny mianownik tych badań (np. skuteczność, skutki uboczne, diagnoza, nowa terapia)?
2. W jakim dokładnie kontekście badane/używane są wymienione leki (z czym je porównywano, na co działają)?

Abstrakty do analizy:
{combined_abstracts}"""

    try:
        response = client.chat.completions.create(
            model=MODEL_NAME,
            messages=[
                {"role": "system", "content": "Jesteś ekspertem w analizie publikacji medycznych. Odpowiadasz precyzyjnie, w oparciu o dostarczony tekst."},
                {"role": "user", "content": prompt}
            ],
            temperature=0.3,
            max_tokens=1200
        )
        return response.choices[0].message.content
    except Exception as e:
        print(f"Błąd przy temacie {topic_id}: {e}")
        return "Błąd generowania"

In [ ]:
summaries = []
grouped_abstracts = df_valid.groupby('Topic')

for index, row in results_df.iterrows():
    topic_id = row['Topic']
    top_drugs = row['Top_Drugs']
    info_row = topic_info[topic_info["Topic"] == topic_id].iloc[0]
    topic_name = info_row["Name"]
    rep = info_row["Representation"]
    keywords = ", ".join(w for w in (rep if isinstance(rep, list) else str(rep).split(", ")) if w)

    if topic_id in grouped_abstracts.groups:
        topic_texts = grouped_abstracts.get_group(topic_id)['Abstract'].astype(str).tolist()

        summary_text = generate_topic_summary(topic_texts, top_drugs, topic_id, topic_name, keywords)

        summaries.append({
            'Topic': topic_id,
            'Topic_Name': topic_name,
            'Top_Drugs': top_drugs,
            'Number_of_Articles': len(topic_texts),
            'AI_Summary': summary_text
        })

        time.sleep(10)

final_report_df = pd.DataFrame(summaries)
final_report_df.to_excel("ms_topics_final_report.xlsx", index=False)

## Interaktywna analiza pomysłów użytkownika
W tej sekcji sprawdzamy, czy pomysł podany przez użytkownika jest zgodny z aktualnymi trendami badawczymi w stwardnieniu rozsianym, a jeśli tak, to z jakim prawdopodobieństwem.

In [ ]:
import json

def is_ms_related(user_text, client):
    prompt = f"""Oceń, czy poniższy tekst (abstrakt naukowy lub pomysł badawczy) jest związany ze stwardnieniem rozsianym (Multiple Sclerosis), demielinizacją, neurologią lub terapiami stosowanymi w tej chorobie.
Odrzuć teksty niezwiązane z tematem (np. teoria gier, finanse, inne dziedziny niezwiązane z medycyną/MS).

Zwróć WYŁĄCZNIE poprawny obiekt JSON w tym formacie:
{{"is_valid": true lub false, "reason": "krótkie, jednozdaniowe uzasadnienie po polsku"}}

Tekst do oceny:
{user_text[:1500]}"""

    try:
        response = client.chat.completions.create(
            model= "openai/gpt-oss-120b",
            messages=[
                {"role": "system", "content": "Jesteś rygorystycznym klasyfikatorem domenowym. Odpowiadasz wyłącznie w formacie JSON."},
                {"role": "user", "content": prompt}
            ],
            temperature=0.0,
            response_format={"type": "json_object"},
            max_tokens=150
        )
        return json.loads(response.choices[0].message.content)
    except Exception as e:
        print(f"Błąd walidacji: {e}")
        return {"is_valid": False, "reason": "Nie udało się zweryfikować tekstu."}

In [ ]:
def analyze_user_idea(user_text, client, topic_model, final_report_df, top_n=3, threshold=0.35):
    validation = is_ms_related(user_text, client)

    if not validation.get("is_valid"):
        return {
            "status": "rejected",
            "message": f"Tekst odrzucony: {validation.get('reason', 'Brak powodu')}"
        }

    similar_topics, similarities = topic_model.find_topics(user_text, top_n=top_n)

    matches = []
    for topic_id, score in zip(similar_topics, similarities):
        if topic_id == -1 or score < threshold:
            continue

        matching_rows = final_report_df[final_report_df['Topic'] == topic_id]
        if not matching_rows.empty:
            trend_info = matching_rows.iloc[0]
            matches.append({
                "topic_id": int(topic_id),
                "similarity_score": round(float(score), 4),
                "topic_name": trend_info['Topic_Name'],
                "top_drugs": trend_info['Top_Drugs'],
                "trend_summary": trend_info['AI_Summary']
            })

    if not matches:
        best_score = max(similarities) if len(similarities) > 0 else 0.0
        return {
            "status": "outlier",
            "message": f"Tekst dotyczy MS, ale nie pasuje wystarczająco do obecnych trendów (najwyższe podobieństwo: {best_score:.2f})."
        }

    return {
        "status": "success",
        "matches": matches
    }

In [ ]:
test_text = "Ocrelizumab reduces brain atrophy in RRMS patients over 3 years on MRI."
wynik = analyze_user_idea(test_text, client, topic_model, final_report_df, top_n=3, threshold=0.30)

if wynik["status"] == "success":
    print(f"Znaleziono {len(wynik['matches'])} pasujące trendy:\n")
    for i, match in enumerate(wynik["matches"], 1):
        print(f"{i}. [{match['topic_id']}] {match['topic_name']} (podobieństwo: {match['similarity_score']})")
        print(f"   Leki: {match['top_drugs']}")
        print(f"   Podsumowanie: {match['trend_summary'][:200]}...\n")
else:
    print(f"[{wynik['status'].upper()}] {wynik['message']}")